# Fase 1 — Extracción de Datos

Conexión a BigQuery y construcción del Dataset Maestro de Rentabilidad.

In [1]:
from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account
from pathlib import Path
import os
import pandas as pd
from sqlalchemy import create_engine

load_dotenv(Path.cwd().parent / ".env", override = True)

client = bigquery.Client(
    credentials=service_account.Credentials.from_service_account_file(
        os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
    ),
    project=os.getenv("BIGQUERY_PROJECT_ID")
)

print("CONEXION CORRECTA, VERIFICAR LOGS")

CONEXION CORRECTA, VERIFICAR LOGS


¿Qué productos y categorías están destruyendo el flujo de caja de la empresa debido a su alta edad en inventario y su baja rentabilidad real tras devoluciones?

In [24]:
'''os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
engine = create_engine(f'bigquery://{os.getenv("BIGQUERY_PROJECT_ID")}')

query = """
WITH inventory_base AS (
    SELECT 
        ii.id AS inventory_item_id,
        ii.product_id,
        p.category,
        p.brand,
        ii.cost,
        oi.sale_price,
        oi.status,
        ii.created_at AS inv_created_at,
        ii.sold_at,
        oi.returned_at,
        TIMESTAMP_DIFF(COALESCE(ii.sold_at, CURRENT_TIMESTAMP()), ii.created_at, DAY) AS days_in_inventory
    FROM `bigquery-public-data.thelook_ecommerce.inventory_items` ii
    INNER JOIN `bigquery-public-data.thelook_ecommerce.products` p 
        ON ii.product_id = p.id
    LEFT JOIN `bigquery-public-data.thelook_ecommerce.order_items` oi 
        ON ii.id = oi.inventory_item_id
),
inventory_metrics AS (
    SELECT 
        *,
        (sale_price - cost) AS unit_margin,
        AVG(days_in_inventory) OVER(PARTITION BY category) AS cat_avg_days_in_inv,
        PERCENT_RANK() OVER(PARTITION BY category ORDER BY cost) AS cost_percentile_in_cat,
        COUNT(inventory_item_id) OVER(PARTITION BY product_id) AS total_sku_stock_history,
        CASE WHEN sold_at IS NULL AND TIMESTAMP_DIFF(CURRENT_TIMESTAMP(), inv_created_at, DAY) > 180 THEN 1 ELSE 0 END AS is_dead_stock
    FROM inventory_base
)
SELECT 
    *,
    SAFE_DIVIDE(days_in_inventory, cat_avg_days_in_inv) AS inventory_age_index
FROM inventory_metrics
"""

df_master = pd.read_sql(query, engine)
df_master.to_parquet('../data/raw_master_data.parquet', index=False) # guardamos localmente la consulta

df_master.info()
'''
print("Consulta realizada y guardada")
# Ejecutamos solo una vez, ya que hacer la consulta cada que se corre el notebook consume tiempo y cuota de google

Consulta realizada y guardada


In [3]:
df_master = pd.read_parquet("../data/raw_master_data.parquet")

# ANALISIS BREVE DE QUERY IMPLEMENTADA
USO: JOINS, CTEs Y WINDOWS FUNCTIONS aprovechando la velocidad e indexacion que da SQL

- inventory_base: construye una tabla base de inventario que incluye valores de la tabla principal (inventory_item) pero enriquecido con la tabla products (categoria y marca) asi como tambien order_items (precio de venta, estado y devoluciones). Ademas calcula cuantos dias estuve (o sigue) en inventario, desde su creacion hasta su venta o hasta hoy en caso nunca se vendio

- inventory_metrics: Sobre la CTE anterior, precalcula algunas metricas como ganancia por unidad (unit_margin), promedio de dias en el inventario por categoria (cat_avg_days_in_inv), en que percentil cae el costo del producto dentro de su categoria (cost_percentil_in_cat), total de unidades en toda la historia que tuvo el producto (total_sku_stock_history) y finalmente un indicador muy importe para la investigacion de este proyecto, is_dead_stock indica si el stock esta muerto si no se vendio o lleva mas de 180 dias sin venderse

- Select final: Agrega una metrica con el nombre inventory_age_index que se calcula days_in_inventory/promedio_categoria, da informacion relevante ya que si >1 indica rota mas lento en su categoria, <1 rota mas rapido en su categoria y cercano a 1, tiene un comportamiento promedio

In [4]:
df_master = df_master
df_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487616 entries, 0 to 487615
Data columns (total 17 columns):
 #   Column                   Non-Null Count   Dtype              
---  ------                   --------------   -----              
 0   inventory_item_id        487616 non-null  int64              
 1   product_id               487616 non-null  int64              
 2   category                 487616 non-null  object             
 3   brand                    487275 non-null  object             
 4   cost                     487616 non-null  float64            
 5   sale_price               180723 non-null  float64            
 6   status                   180723 non-null  object             
 7   inv_created_at           487616 non-null  datetime64[ns, UTC]
 8   sold_at                  180723 non-null  datetime64[ns, UTC]
 9   returned_at              17953 non-null   datetime64[ns, UTC]
 10  days_in_inventory        487616 non-null  int64              
 11  unit_margin  

In [5]:
check_cols = ['inventory_item_id', 'product_id', 'cost', 'inv_created_at']
print(f"Valores nulos en columnas criticas:\n{df_master[check_cols].isnull().sum()}")

print(f"\nDistribucion de estados de orden:\n{df_master['status'].value_counts(dropna=False)}")

print(f"\nConteo de Stock Muerto identificado en SQL: {df_master['is_dead_stock'].sum()}")

Valores nulos en columnas criticas:
inventory_item_id    0
product_id           0
cost                 0
inv_created_at       0
dtype: int64

Distribucion de estados de orden:
status
None          306893
Shipped        54337
Complete       45387
Processing     35768
Cancelled      27278
Returned       17953
Name: count, dtype: int64

Conteo de Stock Muerto identificado en SQL: 283264


# BREVE ANALISIS FINAL DEL NOTEBOOK
Este notebook se centro en la extracción de la data mediante la api y project de google para BIGQUERY. Cuando la conexion es exitosa, realizamos una consulta (optimizada para nuestra pregunta objetivo) al data warehouse que brinda google, usando `sqlalchemy`

Luego de realizar la consulta, guardamos los datos en un dataframe de pandas con el nombre `df_master` y el paso mas importante es exportarlo como archivo .parquet, esto nos permite no hacer consulta constantemente la base de datos y guardar de manera local los datos manteniendo su tipo.

En la celda final revisamos brevemente la integridad de los datos, contabilizando nulos en columnas criticas.